# 05 Evaluation Report — понятный отчёт по текущему прогону

Эта тетрадка ничего нового не обучает и не считает с нуля тяжёлые модели. Она собирает результаты предыдущих тетрадок в один читаемый отчёт.

Её удобно открывать, когда нужно быстро понять:

- разметка вообще валидная или нет;
- сколько у нас точных дублей, разных фасовок и разных товаров;
- какой простой метод сейчас лучше;
- где методы ошибаются опасно;
- что происходит при сборке групп.


## Как читать этот отчёт

Не надо смотреть только на одну итоговую цифру. Для SKU matching цена ошибок несимметрична.

Главный production-критерий для auto-merge:

1. `auto_same_precision` должен быть не ниже целевого порога.
2. `false_merge_count` на `dev` должен проходить лимит безопасности.
3. Среди безопасных threshold-ов выбирается максимальный `auto_same_recall`.

`macro_f1` можно читать как вспомогательную forced-метрику, но он не выбирает production threshold: false merge разных товаров намного хуже, чем пропуск дубля в `manual_review`.


## Блок кода 1. Подготовка окружения

Эта ячейка подключает библиотеки, находит корень проекта и настраивает вывод таблиц.

Здесь нет бизнес-логики: это техническая подготовка, чтобы следующие ячейки могли читать CSV и красиво показывать таблицы.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

from IPython.display import Markdown, display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)


## Блок кода 2. Загрузка всех готовых результатов

Эта ячейка читает файлы, созданные предыдущими этапами:

- разметку из `labeling_sauces.csv`;
- cost-sensitive calibration из `artifacts/reports/threshold_calibration_dev.csv`;
- финальную test-проверку из `artifacts/reports/threshold_evaluation_test.csv`;
- опасные ошибки из `artifacts/reports/false_merges_on_test.csv`;
- результаты графовой сборки из `clustering_pair_eval_sauces.csv` и `clustering_components_sauces.csv`.

Если здесь ошибка про отсутствующий файл, значит сначала надо выполнить `03` и `04`.


In [ ]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"
LABELING_PATH = DATA_DIR / "labeling_sauces.csv"
MATCHING_SUMMARY_PATH = DATA_DIR / "matching_summary_sauces.csv"
THRESHOLD_CALIBRATION_PATH = REPORTS_DIR / "threshold_calibration_dev.csv"
THRESHOLD_EVALUATION_PATH = REPORTS_DIR / "threshold_evaluation_test.csv"
FALSE_MERGES_TEST_PATH = REPORTS_DIR / "false_merges_on_test.csv"
MANUAL_REVIEW_TEST_PATH = REPORTS_DIR / "manual_review_pairs_test.csv"
CLUSTERING_PAIR_EVAL_PATH = DATA_DIR / "clustering_pair_eval_sauces.csv"
CLUSTERING_COMPONENTS_PATH = DATA_DIR / "clustering_components_sauces.csv"

required_paths = [
    LABELING_PATH,
    THRESHOLD_CALIBRATION_PATH,
    THRESHOLD_EVALUATION_PATH,
    FALSE_MERGES_TEST_PATH,
    MANUAL_REVIEW_TEST_PATH,
    CLUSTERING_PAIR_EVAL_PATH,
    CLUSTERING_COMPONENTS_PATH,
]
missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Run notebooks 03 and 04 first. Missing: " + ", ".join(str(path) for path in missing))

labels = pd.read_csv(LABELING_PATH)
matching_summary = pd.read_csv(MATCHING_SUMMARY_PATH) if MATCHING_SUMMARY_PATH.exists() else pd.DataFrame()
calibration_on_dev = pd.read_csv(THRESHOLD_CALIBRATION_PATH)
evaluation_on_test = pd.read_csv(THRESHOLD_EVALUATION_PATH)
false_merges = pd.read_csv(FALSE_MERGES_TEST_PATH)
manual_review_test = pd.read_csv(MANUAL_REVIEW_TEST_PATH)
pair_eval = pd.read_csv(CLUSTERING_PAIR_EVAL_PATH)
components = pd.read_csv(CLUSTERING_COMPONENTS_PATH)

print(f"Loaded labels: {len(labels)} rows")
print(f"Loaded dev calibration: {len(calibration_on_dev)} rows")
print(f"Loaded test evaluation: {len(evaluation_on_test)} rows")
print(f"Loaded clustering pair eval: {len(pair_eval)} rows")


## Блок кода 3. Проверка разметки

Эта ячейка отвечает на вопрос: можно ли вообще доверять CSV с ручной разметкой как входу для метрик.

Смотри:

- `empty_labels` — пустые метки, их быть не должно;
- `invalid_labels` — неизвестные метки, их тоже быть не должно;
- распределение классов — сколько пар каждого типа;
- разрез по маркетплейсам — хватает ли межмаркетплейсных пар.

Если разметка грязная, любые метрики ниже будут сомнительными.


In [ ]:
allowed_labels = ["exact_duplicate", "different_product", "uncertain"]
label_values = labels["label"].fillna("").astype(str).str.strip()
label_distribution = label_values.value_counts().rename_axis("label").reset_index(name="pairs")
label_distribution["share"] = label_distribution["pairs"] / len(labels)
invalid_labels = sorted(set(label_values) - set(allowed_labels) - {""})
empty_labels = int(label_values.eq("").sum())

qa_summary = pd.DataFrame([
    {"check": "rows", "value": len(labels)},
    {"check": "empty_labels", "value": empty_labels},
    {"check": "invalid_labels", "value": len(invalid_labels)},
    {"check": "uncertain_pairs", "value": int(label_values.eq("uncertain").sum())},
])

display(qa_summary)
display(label_distribution)
if invalid_labels:
    display(Markdown("Invalid labels: " + ", ".join(invalid_labels)))

if "is_cross_marketplace_pair" in labels.columns:
    display(pd.crosstab(labels["is_cross_marketplace_pair"], label_values))


## Блок кода 4. Поиск подозрительных мест в разметке

Эта ячейка не исправляет разметку автоматически. Она только подсвечивает пары, которые стоит проверить глазами.

Например:

- legacy `same_product_different_pack` labels, которые нужно заменить на `exact_duplicate` перед бинарными метриками;
- `exact_duplicate`, где pack signature отличается: это уже не ошибка label, а ожидаемый вход для pack-правил;
- `different_product`, но модельный поиск нашёл очень высокое сходство и фасовка совпадает.

Такие строки не обязательно ошибки. Это список для ручного контроля качества.


In [ ]:
work = labels.copy()
for column in ["unit_amount_a", "unit_amount_b", "multipack_count_a", "multipack_count_b"]:
    if column in work.columns:
        work[column] = pd.to_numeric(work[column], errors="coerce")

same_unit = (work["unit_amount_a"] - work["unit_amount_b"]).abs().le(0.02)
same_pack = (work["multipack_count_a"] - work["multipack_count_b"]).abs().le(0.25)
pack_signature_same = same_unit & same_pack
label_norm = label_values

sanity_rows = pd.DataFrame([
    {"check": "legacy_same_product_different_pack_labels", "pairs": int(label_norm.eq("same_product_different_pack").sum())},
    {"check": "exact_duplicate_pack_signature_differs", "pairs": int(((label_norm == "exact_duplicate") & ~pack_signature_same).sum())},
    {"check": "different_product_same_pack_high_embedding", "pairs": int(((label_norm == "different_product") & pack_signature_same & work["embedding_similarity_score"].ge(0.95)).sum())},
])
display(sanity_rows)

display(Markdown("Sanity rows above are not automatic errors. They mark pairs worth revisiting if final metrics look unstable."))


## Блок кода 5. Сводная таблица качества методов

Эта ячейка показывает главную таблицу сравнения.

Как читать:

- `threshold_calibration_dev.csv` — единственное место, где выбираются пороги и метод-кандидат;
- `passed_auto_same_constraints` — прошла ли модель safety-условия для auto-merge;
- `manual_review_rate` — какая доля пар не получила автоматическое решение;
- `threshold_evaluation_test.csv` — только финальная проверка уже выбранных на `dev` порогов.

`test` не используется для выбора метода. Он нужен только для честной проверки результата.


In [ ]:
dev_ranked = calibration_on_dev.sort_values(
    ["passed_auto_same_constraints", "auto_coverage", "auto_same_recall", "manual_review_rate"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
test_view = evaluation_on_test.sort_values(
    ["passed_auto_same_constraints", "auto_coverage", "auto_same_precision"],
    ascending=[False, False, False],
).reset_index(drop=True)

display(Markdown("### Calibration on dev"))
display(dev_ranked)
display(Markdown("### Evaluation on test"))
display(test_view)

if dev_ranked.empty:
    selected_method = None
    display(Markdown("No dev calibration summary found."))
else:
    selected = dev_ranked.iloc[0]
    selected_method = selected["method"]
    status = "passed" if bool(selected["passed_auto_same_constraints"]) else "did not pass"
    display(Markdown(
        f"Selected current baseline by **dev calibration**: **{selected_method}** ({status} auto-merge constraints), "
        f"auto coverage **{selected['auto_coverage']:.1%}**, manual review **{selected['manual_review_rate']:.1%}**."
    ))


## Блок кода 6. Конкретные опасные ошибки

Эта ячейка выводит пары, где выбранный метод сделал false merge на `test`.

Это значит: `same_base_product=0`, но triage решил `auto_same`. Такие ошибки самые дорогие, поэтому они читаются отдельно от общей метрики.


In [ ]:
if selected_method is None:
    display(false_merges.head(0))
else:
    test_false_merges = false_merges[false_merges["method"].eq(selected_method)].copy()
    selected_manual = manual_review_test[manual_review_test["method"].eq(selected_method)].copy()
    display(Markdown(
        f"Test false merges for **{selected_method}**: {len(test_false_merges)}; "
        f"manual review pairs: {len(selected_manual)}"
    ))
    columns = [
        "label",
        "same_base_product",
        "predicted_triage_label",
        "score",
        "threshold_auto_same",
        "threshold_auto_diff",
        "title_a",
        "title_b",
        "brand_a",
        "brand_b",
        "unit_amount_a",
        "unit_amount_b",
        "multipack_count_a",
        "multipack_count_b",
    ]
    display(test_false_merges[[column for column in columns if column in test_false_merges.columns]].head(15))


## Блок кода 7. Качество группировки

Эта ячейка берёт результат тетрадки `04` и кратко показывает качество графа.

`family` — проверка базовых товарных семей по binary positive labels.

`pack` — проверка конкретных фасовок после deterministic pack-правил.

`false_links` — лишние связи в графе. Они опасны, потому что могут склеить разные товары в одну группу.

`missed_links` — пропущенные связи. Они означают, что часть дублей осталась раздельно.


In [ ]:
def _binary_link_report(frame: pd.DataFrame, *, true_col: str, pred_col: str, scope: str) -> dict[str, object]:
    true_link = frame[true_col].astype(bool)
    pred_link = frame[pred_col].astype(bool)
    tp = int((true_link & pred_link).sum())
    fp = int((~true_link & pred_link).sum())
    fn = int((true_link & ~pred_link).sum())
    tn = int((~true_link & ~pred_link).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "scope": scope,
        "pairs": len(frame),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "false_links": fp,
        "missed_links": fn,
        "true_positive_links": tp,
        "true_negative_links": tn,
    }

heldout_pairs = pair_eval[pair_eval["eval_split"].eq("test")].copy()
cluster_report = pd.DataFrame([
    _binary_link_report(heldout_pairs, true_col="true_same_family", pred_col="pred_same_family", scope="family"),
    _binary_link_report(heldout_pairs, true_col="true_same_pack", pred_col="pred_same_pack", scope="pack"),
])
display(cluster_report)

component_summary = pd.DataFrame([
    {"graph": "pred_family", "components": components["pred_family_id"].nunique(), "multi_node_components": int((components.groupby("pred_family_id").size() > 1).sum())},
    {"graph": "pred_pack", "components": components["pred_pack_id"].nunique(), "multi_node_components": int((components.groupby("pred_pack_id").size() > 1).sum())},
    {"graph": "true_family_partial", "components": components["true_family_id"].nunique(), "multi_node_components": int((components.groupby("true_family_id").size() > 1).sum())},
    {"graph": "true_pack_partial", "components": components["true_pack_id"].nunique(), "multi_node_components": int((components.groupby("true_pack_id").size() > 1).sum())},
])
display(component_summary)


## Блок кода 8. Текущие выводы человеческим языком

Эта ячейка собирает короткий список выводов по текущему прогону.

Её удобно использовать как черновик для защиты или для следующего обсуждения: что уже доказали, что пока плохо, и какой следующий метод нужен.


In [ ]:
selected_status = "нет выбранного метода"
if selected_method is not None:
    selected_status = f"выбранный по dev calibration метод: {selected_method}"

passed_count = int(calibration_on_dev["passed_auto_same_constraints"].astype(bool).sum()) if not calibration_on_dev.empty else 0
manual_review_note = "нет test evaluation"
if selected_method is not None and not evaluation_on_test.empty:
    selected_test = evaluation_on_test[evaluation_on_test["method"].eq(selected_method)]
    if not selected_test.empty:
        manual_review_note = f"manual_review на test: {selected_test.iloc[0]['manual_review_rate']:.1%}"

conclusions = [
    "Gold-set используется как binary same_base_product: same base product vs different product.",
    f"Auto-merge выбирается cost-sensitive правилом на dev; {passed_count} method(s) прошли safety constraints.",
    f"{selected_status}; {manual_review_note}.",
    "Macro-F1 оставлен как вспомогательная forced-метрика, но не выбирает threshold или модель.",
    "Следующий шаг: читать false_merges/manual_review пары и улучшать reranker или правила post-processing без подгонки под test.",
]

body = "## Current conclusions\n" + "\n".join(f"- {item}" for item in conclusions)
display(Markdown(body))


## Что делать после отчёта

Этот отчёт показывает, какие методы безопасны для auto-merge при заданных dev-ограничениях.

Следующие практические шаги:

1. Глазами проверить `false_merges_on_test.csv` и `manual_review_pairs_test.csv`.
2. Ослаблять/ужесточать production-критерии только через явные параметры calibration, а не через test.
3. Улучшать reranker/cross-encoder на hard negatives и спорных `manual_review` случаях.
4. Только после выбора безопасного метода запускать его по полному списку кандидатов.
